# Part 6 — RAG Final Evaluation Assessment

**Aim:** Check retrieval relevance, factual correctness, completeness and source support. Reuse the existing Part 5 pipeline and its **top-5** retrieval setting. Do not rebuild the index.

**Method:** Review retrieved evidence → verify reference papers and expected facts → optionally generate answers → manually score → calculate only verified results. Based on evaluation principles discussed in [EnterpriseRAG-Bench](https://arxiv.org/abs/2605.05253); this is a small student assessment, **not** a reproduction of that benchmark.

Run this section **after loading Part 5 variables** (`processed_dir`, `rag_retriever`, `build_research_context`, `generate_business_insight`, `llm`). The original historical questions cannot be used as gold-standard tests against the rebuilt corpus because their papers are absent.

In [ ]:
from pathlib import Path
import pandas as pd

needed = ['processed_dir', 'rag_retriever', 'build_research_context', 'generate_business_insight', 'llm']
missing = [x for x in needed if x not in globals()]
if missing:
    raise RuntimeError('Load Part 5 first: ' + ', '.join(missing))

OUT = Path(processed_dir) / 'rag_final_evaluation'
OUT.mkdir(parents=True, exist_ok=True)
TOP_K = 5
RUN_GENERATION = False  # Set True only after approving Groq API use.
print('Evaluation folder:', OUT)


## 6.1 Questions and retrieval

Questions are **candidates**, not verified gold labels. The last question tests appropriate acknowledgement of missing business evidence. The retrieved passages are saved for inspection; a similarity score is not a measure of factual accuracy.

In [ ]:
questions = [
    ('Q1', 'What methods can reduce hallucinations in retrieval-augmented generation?', 'answerable'),
    ('Q2', 'How do dense and sparse retrieval approaches differ in strengths and limitations?', 'answerable'),
    ('Q3', 'How can the factual support of RAG answers be evaluated?', 'answerable'),
    ('Q4', 'What risks should businesses consider when deploying RAG assistants?', 'answerable'),
    ('Q5', 'What are the main failure modes of retrieval-augmented generation?', 'answerable'),
    ('Q6', 'What exact financial savings has KPMG achieved from this student prototype?', 'missing_evidence'),
]
qdf = pd.DataFrame(questions, columns=['id', 'question', 'type'])
qdf.to_csv(OUT / 'questions.csv', index=False)

hits, rows = {}, []
for q in qdf.itertuples(index=False):
    found = rag_retriever.retrieve(q.question)[:TOP_K]
    hits[q.id] = found
    for rank, item in enumerate(found, 1):
        meta = item.node.metadata or {}
        rows.append({'id': q.id, 'rank': rank, 'node_id': item.node.node_id,
                     'paper_id': str(meta.get('arxiv_id', '')),
                     'title': str(meta.get('title', '')),
                     'page': meta.get('page_number', ''),
                     'score': item.score, 'passage': item.node.text})
passages = pd.DataFrame(rows, columns=['id','rank','node_id','paper_id','title','page','score','passage'])
passages.to_csv(OUT / 'retrieved_passages.csv', index=False)
print('Questions:', len(qdf), '| Retrieved passages:', len(passages))
display(passages[['id','rank','paper_id','title','score']])


## 6.2 Verify evidence and review answers

Before scoring, open the source papers and complete `reference_review.csv`: set `verified=yes`, list **independently verified** relevant paper IDs (comma-separated) and expected facts. Leave unknown fields blank. Do not assume the retrieved top five contain all relevant papers.

Generation is optional and disabled by default. It uses the **unchanged** Part 5 prompt and may consume Groq API quota.

In [ ]:
reference_path = OUT / 'reference_review.csv'
if not reference_path.exists():
    ref = qdf.copy()
    ref['verified'] = ''
    ref['relevant_paper_ids'] = ''
    ref['expected_facts'] = ''
    ref.to_csv(reference_path, index=False)

answer_path = OUT / 'answers.csv'
if RUN_GENERATION:
    if answer_path.exists():
        raise FileExistsError('answers.csv already exists; review it before making more API calls.')
    ref = pd.read_csv(reference_path, dtype=str).fillna('')
    approved = set(ref.loc[(ref['type'] == 'answerable') &
                            (ref['verified'].str.lower() == 'yes') &
                            ref['relevant_paper_ids'].str.strip().ne(''), 'id'])
    approved |= set(qdf.loc[qdf['type'] == 'missing_evidence', 'id'])
    output = []
    for q in qdf.itertuples(index=False):
        if q.id not in approved or not hits[q.id]:
            continue
        try:
            context = build_research_context(hits[q.id], max_sources=TOP_K)
            answer = generate_business_insight(question=q.question, context=context, llm=llm)
        except Exception as exc:
            print('Stopped after error on', q.id, type(exc).__name__)
            break  # No automatic retries.
        output.append({'id': q.id, 'answer': str(answer)})
        pd.DataFrame(output).to_csv(answer_path, index=False)
    print('Saved answers:', len(output))
else:
    print('Generation off. Verify evidence first, then explicitly enable if needed.')

review_path = OUT / 'human_review.csv'
if not review_path.exists():
    review = qdf[['id', 'type']].copy()
    for col in ['retrieval_relevance', 'correctness', 'completeness', 'source_support']:
        review[col] = ''  # Human scores: 0=poor, 1=partial, 2=good.
    review['correct_abstention'] = ''  # yes/no for missing_evidence only
    review['notes'] = ''
    review.to_csv(review_path, index=False)
print('Review files:', reference_path.name, review_path.name)


## 6.3 Results and limitations

For answerable questions, score each dimension **0 / 1 / 2** after reading the retrieved passages and generated answer. Include only questions with verified reference evidence, saved answers and all four human scores. For the missing-evidence case, mark `correct_abstention=yes` only if the model explicitly refuses to invent a savings figure. **Blank means not reviewed**, never zero.

`Known-paper Hit@5` measures whether at least one independently verified relevant paper was retrieved. It does not measure complete corpus recall.

In [ ]:
ref = pd.read_csv(reference_path, dtype=str).fillna('')
review = pd.read_csv(review_path, dtype=str).fillna('')
answers = (pd.read_csv(answer_path, dtype=str).fillna('')
           if answer_path.exists() else pd.DataFrame(columns=['id','answer']))
scored = qdf.merge(ref[['id','verified','relevant_paper_ids']], on='id').merge(review, on=['id','type'])
scored = scored[scored['type'] == 'answerable'].copy()
scored = scored[(scored['verified'].str.lower() == 'yes') &
                scored['relevant_paper_ids'].str.strip().ne('') &
                scored['id'].isin(answers['id'])]
metrics = ['retrieval_relevance','correctness','completeness','source_support']
for m in metrics:
    scored[m] = pd.to_numeric(scored[m], errors='coerce')
    if not scored[m].dropna().isin([0,1,2]).all():
        raise ValueError('Invalid human score in ' + m)
scored = scored.dropna(subset=metrics)

summary = pd.DataFrame([{'metric': m, 'reviewed': len(scored),
                         'mean_out_of_2': round(scored[m].mean(), 2) if len(scored) else None}
                        for m in metrics])
summary.to_csv(OUT / 'summary.csv', index=False)

def paper_ids(value):
    return {x.strip() for x in str(value).split(',') if x.strip()}

hit_rows = []
for r in scored.itertuples(index=False):
    gold = paper_ids(r.relevant_paper_ids)
    retrieved = paper_ids(','.join(passages.loc[passages['id'] == r.id, 'paper_id'].fillna('')))
    hit_rows.append({'id': r.id, 'known_paper_hit_at_5': int(bool(gold & retrieved))})
hit_df = pd.DataFrame(hit_rows, columns=['id','known_paper_hit_at_5'])
hit_df.to_csv(OUT / 'known_paper_hits.csv', index=False)

missing = review[review['type'] == 'missing_evidence']
missing = missing[missing['id'].isin(answers['id'])]
missing = missing[missing['correct_abstention'].str.lower().isin(['yes','no'])]
print('Human-reviewed answerable questions:', len(scored))
print('Known-paper Hit@5:', round(hit_df['known_paper_hit_at_5'].mean(), 2) if len(hit_df) else 'Not evaluated')
print('Missing-evidence cases reviewed:', len(missing))
print('Correct abstentions:', int(missing['correct_abstention'].str.lower().eq('yes').sum()) if len(missing) else 'Not evaluated')
display(summary)
print('Limitations: small sample; manually verified labels; arXiv rather than KPMG internal documents;')
print('PDF extraction quality varies; this does not establish production reliability.')
